In [2]:
"""
================================================================================
PROJECT: Stats NZ Labour Market Data Warehouse
FILE NAME: 02_create_sql_views.py
DESCRIPTION: Creates pre-aggregated database views for Power BI validation 
              and SQL verification with explicit data type casting and 
              non-scientific numeric formatting aligned with the Star Schema.
================================================================================
"""

import os
import sqlite3
import pandas as pd

# Suppress scientific notation in Pandas console outputs
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Direct path to your database
DB_PATH = r"D:\ProjectData\LabourMarket_Gold_DW.db"

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"❌ Database file not found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"🚀 Connecting to Database at: {DB_PATH}")
print("🚀 Creating SQL Verification Views...")

# ------------------------------------------------------------------------------
# 1. VIEW: Dataset Overview Summary (Total Rows & Value Sums by Source)
# ------------------------------------------------------------------------------
print("  --> Dropping and building 'v_audit_dataset_summary'...")
cursor.execute("DROP VIEW IF EXISTS v_audit_dataset_summary;")
cursor.execute("""
CREATE VIEW v_audit_dataset_summary AS
SELECT 
    CAST(d.DatasetCode AS TEXT) AS DatasetCode,
    CAST(d.DatasetName AS TEXT) AS DatasetName,
    CAST(COUNT(f.FactID) AS INTEGER) AS TotalRecords,
    CAST(ROUND(SUM(f.DataValue), 2) AS REAL) AS SumDataValue,
    CAST(ROUND(AVG(f.DataValue), 2) AS REAL) AS AvgDataValue,
    CAST(MIN(dt.FullDate) AS TEXT) AS EarliestDate,
    CAST(MAX(dt.FullDate) AS TEXT) AS LatestDate
FROM FactLabourMarket f
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
JOIN DimDate dt ON f.DateKey = dt.DateKey
GROUP BY d.DatasetCode, d.DatasetName;
""")

# ------------------------------------------------------------------------------
# 2. VIEW: Yearly Trends (Aggregated by Year and Dataset)
# ------------------------------------------------------------------------------
print("  --> Dropping and building 'v_yearly_labour_trends'...")
cursor.execute("DROP VIEW IF EXISTS v_yearly_labour_trends;")
cursor.execute("""
CREATE VIEW v_yearly_labour_trends AS
SELECT 
    CAST(dt.Year AS TEXT) AS Year,
    CAST(d.DatasetCode AS TEXT) AS DatasetCode,
    CAST(COUNT(f.FactID) AS INTEGER) AS TotalRecords,
    CAST(ROUND(SUM(f.DataValue), 2) AS REAL) AS TotalValue,
    CAST(ROUND(AVG(f.DataValue), 2) AS REAL) AS AverageValue
FROM FactLabourMarket f
JOIN DimDate dt ON f.DateKey = dt.DateKey
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
GROUP BY dt.Year, d.DatasetCode;
""")

conn.commit()



🚀 Connecting to Database at: D:\ProjectData\LabourMarket_Gold_DW.db
🚀 Creating SQL Verification Views...
  --> Dropping and building 'v_audit_dataset_summary'...
  --> Dropping and building 'v_yearly_labour_trends'...


In [3]:
# ------------------------------------------------------------------------------
# 3. VERIFY RESULTS IN PYTHON CONSOLE
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("📊 VIEW VERIFICATION & DATA BENCHMARKS")
print("=" * 80)

print("\n[VERIFICATION 1] v_audit_dataset_summary Results:")
summary_df = pd.read_sql("SELECT * FROM v_audit_dataset_summary", conn)
print(summary_df.to_string(index=False))

print("\n[VERIFICATION 2] Sample Recent Records from v_yearly_labour_trends:")
trends_df = pd.read_sql("SELECT * FROM v_yearly_labour_trends ORDER BY Year DESC, DatasetCode LIMIT 10", conn)
print(trends_df.to_string(index=False))

conn.close()

print("\n" + "=" * 80)
print("✅ VIEWS SUCCESSFULLY CREATED AND VERIFIED IN SQLITE!")
print("=" * 80)


📊 VIEW VERIFICATION & DATA BENCHMARKS

[VERIFICATION 1] v_audit_dataset_summary Results:
DatasetCode                       DatasetName  TotalRecords    SumDataValue  AvgDataValue EarliestDate LatestDate
       HLFS     Household Labour Force Survey       1207728    206960545.80        171.36   1986-03-31 2026-06-30
        LCI                 Labour Cost Index         32926     28491939.40        865.33   1989-06-30 2026-06-30
        LMS Labour Market Statistics Overview       1438668 877848757685.52     610181.61   1986-03-31 2026-06-30
        MEI     Monthly Employment Indicators         34532   5208798090.98     150839.75   1999-04-30 2026-07-31
        QES       Quarterly Employment Survey        198014 877613305200.32    4432077.05   1989-03-31 2026-06-30

[VERIFICATION 2] Sample Recent Records from v_yearly_labour_trends:
Year DatasetCode  TotalRecords     TotalValue  AverageValue
2026        HLFS         26030     4579589.20        175.94
2026         LCI           784      9